In [1]:
!pip install sqlalchemy psycopg2-binary

In [2]:
from sqlalchemy import create_engine, text

In [3]:
engine = create_engine(
    "postgresql+psycopg2://postgres:PASSWORDlocalhost:5432/TEST"
)

In [4]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print(result.scalar())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 61: invalid continuation byte

In [ ]:
import pandas as pd

query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)

tables

In [ ]:
query = """
SELECT *
FROM orders
LIMIT 10;
"""

orders_sample = pd.read_sql(query, engine)

orders_sample

In [ ]:
query = """
SELECT
    o.order_number,
    o.ordered_at,
    c.client_name,
    m.model_name,
    b.brand_name,
    o.ordered_quantity
FROM orders AS o
JOIN client AS c
    ON o.client_id = c.id
JOIN model AS m
    ON o.model_id = m.id
JOIN brand AS b
    ON m.brand_id = b.id
ORDER BY o.ordered_at;
"""

orders_analysis = pd.read_sql(query, engine)

orders_analysis.head(10)

In [ ]:
query = text("""
SELECT
    o.order_number,
    o.ordered_at,
    c.client_name,
    m.model_name,
    b.brand_name,
    o.ordered_quantity
FROM orders AS o
JOIN client AS c
    ON o.client_id = c.id
JOIN model AS m
    ON o.model_id = m.id
JOIN brand AS b
    ON m.brand_id = b.id
WHERE o.ordered_at >= :start_date
ORDER BY o.ordered_at;
""")

orders_from_february = pd.read_sql(
    query,
    engine,
    params={"start_date": "2026-02-01"}
)

orders_from_february.head()

In [ ]:
brand_analysis = (
    orders_from_february
    .groupby('brand_name')
    .agg(
        total_quantity=('ordered_quantity', 'sum'),
        orders=('order_number', 'nunique')
    )
    .sort_values('total_quantity', ascending=False)
)

brand_analysis['share_pct'] = (
    brand_analysis['total_quantity']
    / brand_analysis['total_quantity'].sum()
    * 100
).round(2)

top3_brands = brand_analysis.head(3)

top3_brands

In [ ]:
query = text("""
SELECT
    b.brand_name,
    SUM(o.ordered_quantity) AS total_quantity,
    COUNT(DISTINCT o.order_number) AS orders
FROM orders AS o
JOIN model AS m
    ON o.model_id = m.id
JOIN brand AS b
    ON m.brand_id = b.id
WHERE o.ordered_at >= :start_date
GROUP BY b.brand_name
ORDER BY total_quantity DESC
LIMIT 3;
""")

top3_sql = pd.read_sql(
    query,
    engine,
    params={"start_date": "2026-02-01"}
)

top3_sql

In [ ]:
query = text("""
SELECT
    o.order_number,
    c.client_name,
    m.model_name,
    b.brand_name,
    os.status_name,
    osh.status_begins,
    osh.status_ends
FROM order_status_history AS osh
JOIN orders AS o
    ON osh.order_id = o.id
JOIN client AS c
    ON o.client_id = c.id
JOIN model AS m
    ON o.model_id = m.id
JOIN brand AS b
    ON m.brand_id = b.id
JOIN order_status AS os
    ON osh.status_id = os.id
ORDER BY o.order_number, osh.status_begins;
""")

status_history = pd.read_sql(query, engine)

status_history.head(10)

In [ ]:
status_history['status_begins'] = pd.to_datetime(status_history['status_begins'], errors='coerce')
status_history['status_ends']   = pd.to_datetime(status_history['status_ends'],   errors='coerce')

In [ ]:
status_history

In [ ]:
status_history['duration'] = (
    status_history['status_ends']
    - status_history['status_begins']
)

In [ ]:
status_history['duration_days'] = (
    status_history['duration'].dt.days
)

In [ ]:
status_history

In [ ]:
status_analysis = (
    status_history
    .dropna(subset=['duration_days'])
    .groupby('status_name')
    .agg(
        avg_duration_days=('duration_days', 'mean'),
        completed_intervals=('duration_days', 'count')
    )
    .sort_values('avg_duration_days', ascending=False)
)

status_analysis

In [ ]:
current_status = (
    status_history
    .sort_values(['order_number', 'status_begins'])
    .groupby('order_number')
    .tail(1)
)

current_status[['order_number', 'client_name', 'model_name', 'brand_name', 'status_name']]

In [ ]:
status_distribution = (
    current_status
    .groupby('status_name')
    .size()
    .to_frame('orders')
)

status_distribution['share_pct'] = (
    status_distribution['orders']
    / status_distribution['orders'].sum()
    * 100
).round(2)

status_distribution = status_distribution.sort_values(
    'orders',
    ascending=False
)

status_distribution

In [ ]:
brand_status = (
    current_status
    .pivot_table(
        index='brand_name',
        columns='status_name',
        values='order_number',
        aggfunc='nunique',
        fill_value=0
    )
)

brand_status

In [ ]:
dominant_status = brand_status.idxmax(axis=1)

dominant_status

In [ ]:
status_history

In [ ]:
status_history = status_history.sort_values(
    ['order_number', 'status_begins']
).copy()

status_history['prev_status'] = (
    status_history.groupby('order_number')['status_name'].shift(1)
)

status_history['prev_end'] = (
    status_history.groupby('order_number')['status_ends'].shift(1)
)

status_history['gap_days'] = (
    status_history['status_begins'] - status_history['prev_end']
).dt.days

In [ ]:
status_history